# Stage 0 — Project setup and assumptions

_Pipeline stage 0 of `docs/PIPELINE_DESIGN.md`. Uses the canonical `channel_heads` package + on-disk artifacts; heavy rebuilds run via the `channel-heads` CLI (see each stage's command)._

## What this project does

We detect **channel-head coupling** in drainage networks — pairs of channel heads
that meet at a confluence whose contributing areas are spatially *touching* — and
ask how common that coupling is on **Mars**, using a classifier **trained on Earth**
(method after *Goren & Shelef 2024*).

Coupling is a fingerprint of **drainage-divide mobility**: when two growing channel
heads share/contest a divide, their basins press together. The coupled fraction of
confluences is a proxy for how dynamic a landscape's divides are — and Earth vs Mars
probes whether martian valley networks froze in a fluvially-active or degraded state.

### The transfer-learning idea
Mars valley networks are larger and lower-resolution than Earth basins, so the model
uses **5 dimensionless features** describing the *geometry* of a confluence
(orientation contrast, normalized head–head distance, apex angle, Strahler-order
difference, a normalized proximity profile). Scale-free geometry lets an Earth-trained
model apply to Mars without rescaling. A small **CNN** adds raster *shape context* via
a 4-D embedding of a 5-class confluence patch.

### Frozen contracts (never change silently)
- Production `xgb_touching_classifier.json` (threshold **0.577406**) and `cnn_outlet_final.pt`.
- The **5 dimensionless features** (order matters) and the **5-class** CNN patch.
- Unit conversions go through `channel_heads.units`.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import channel_heads as ch
from channel_heads.io.paths import PROJECT_ROOT, RESULTS_DIR, EXAMPLE_DEMS
ROOT     = PROJECT_ROOT
MODELS   = ROOT / 'models'
MARS_OUT = ROOT / 'data/Mars/model_outputs'
MARS_IN  = ROOT / 'data/Mars/model_inputs'
REGIMES_ = ['regA', 'regB', 'regC']
# Frozen model features (5 dimensionless) + operating thresholds (Stage 9).
MODEL_FEATURES = ['orientation_diff_deg','headhead_dist_norm','apex_angle_deg',
                  'strahler_order_diff','proximity_profile_norm']
OP_THR = {'regA': 0.756326, 'regB': 0.779264, 'regC': 0.759369}
print('channel_heads', ch.__version__, '| root', ROOT)


channel_heads 0.1.0 | root /Users/guypi/Projects/channel-heads


### Canonical paths, the 17 Earth basins, and the three frozen regimes

In [2]:
from channel_heads.regimes import REGIMES
print('Earth DEMs available:', len(EXAMPLE_DEMS))
print('Frozen model features:', MODEL_FEATURES)
pd.DataFrame([{'regime': r.name, 'threshold_km2': r.threshold_km2,
               'pre_remove_max_order': r.pre_remove_max_order,
               'order_gap_to_prune': r.order_gap_to_prune,
               'character': c} for r, c in zip(
                   REGIMES.values(),
                   ['dense base / aggressive tip removal',
                    'sparse base / minimal pruning',
                    'intermediate density'])])

Earth DEMs available: 17
Frozen model features: ['orientation_diff_deg', 'headhead_dist_norm', 'apex_angle_deg', 'strahler_order_diff', 'proximity_profile_norm']


,regime,threshold_km2,pre_remove_max_order,order_gap_to_prune,character
0,regA,0.05,2,4,dense base / aggressive tip removal
1,regB,0.25,1,4,sparse base / minimal pruning
2,regC,0.10,1,4,intermediate density


The three regimes (regA/B/C) re-run the whole pipeline at different network
*complexities* — the project's way of quantifying **calibration uncertainty**. No
single pruning is "correct"; we report the spread.

In [3]:
# Unit contract demo: the same arc-degree distance differs in metres by latitude.
from channel_heads.units import compute_meters_per_degree
for lat in (10, 36, 60):
    print(f'lat {lat:>2}deg:  1 deg longitude = {compute_meters_per_degree(lat):8.1f} m')

lat 10deg:  1 deg longitude = 110083.5 m
lat 36deg:  1 deg longitude =  99775.8 m
lat 60deg:  1 deg longitude =  78438.9 m


**Why this matters:** a head–head distance in pixels/degrees must be converted with
the basin's latitude before it is physical — and Mars uses a different planetary
radius. Centralizing this in `units.py` keeps Earth and Mars numerically comparable.

Deep dives: `docs/PIPELINE_DESIGN.md`, `docs/REGIME_SELECTION.md`, `docs/modeling.md`.